# 误差反向传播法

In [2]:
import numpy as np

### 简单层的实现

In [3]:
class MulLayer:
    """
    乘法层
    """
    def __init__(self):
        self.x = None
        self.y = None
        
    def forward(self, x, y):
        self.x = x
        self.y = y
        out = x * y
        
        return out
    
    def backward(self, dout):
        dx = dout * self.y
        dy = dout * self.x
        
        return dx, dy

class AddLayer:
    """
    加法层
    """
    def __init__(self):
        pass
    
    def forward(self, x, y):
        out = x + y
        
        return out
    
    def backward(self, dout):
        dx = dout * 1
        dy = dout * 1
        
        return dx, dy


#### 实现"购买2个苹果和3个橘子"


![购买2个苹果和3个橘子](../screen-short/5-17.png)

### 激活函数层实现

In [4]:
# ReLU
class ReluLayer:
    """
    ReLU层
    """
    def __init__(self):
        self.mask = None
        
    def forward(self, x):
        self.mask = (x <= 0)
        out = x.copy()
        out[self.mask] = 0
        
        return out
    
    def backward(self, dout):
        dout[self.mask] = 0
        dx = dout
        
        return dx

In [5]:
# Sigmoid
class SigmoidLayer:
    """
    Sigmoid层
    """
    def __init__(self):
        self.out = None
        
    def forward(self, x):
        self.out = 1 / (1 + np.exp(-x))
        return self.out
    
    def backward(self, dout):
        dx = dout * (1.0 - self.out) * self.out
        return dx

In [6]:
# Softmax with Cross Entropy Loss
class SoftmaxWithLoss:
    """
    Softmax with Cross Entropy Loss
    """
    def __init__(self):
        self.y = None
        self.t = None
        
    def forward(self, x, t):
        self.t = t
        exp_x = np.exp(x - np.max(x))  # 防止溢出
        self.y = exp_x / np.sum(exp_x)
        
        loss = -np.sum(t * np.log(self.y + 1e-7))  # 加上小常数防止log(0)
        
        return loss
    
    def backward(self):
        batch_size = self.t.shape[0]
        dx = (self.y - self.t) / batch_size
        
        return dx

In [7]:
# Affine Layer
class AffineLayer:
    """
    仿射层
    """
    def __init__(self, W, b):
        self.W = W
        self.b = b
        self.x = None
        self.dW = None
        self.db = None
        
    def forward(self, x):
        self.x = x
        out = np.dot(x, self.W) + self.b
        
        return out
    
    def backward(self, dout):
        self.dW = np.dot(self.x.T, dout)
        self.db = np.sum(dout, axis=0)
        dx = np.dot(dout, self.W.T)
        
        return dx